# MediFlow Hair — 정리 데이터 2단계 학습

정리된 `hair_clean_v1` 데이터로 EfficientNet-B0를 새로 학습합니다. 기존 Hair 모델은 이전 분할의 영향을 받았을 수 있으므로 초기 모델로 사용하지 않습니다.

- 1단계: ImageNet EfficientNet-B0를 고정하고 분류 부분만 15 Epoch 학습
- 2단계: 1단계 최고 모델의 Backbone 마지막 30개 계층 중 Batch Normalization을 제외한 계층을 10 Epoch 추가 학습
- 고정 조건: 데이터 분할, 클래스 순서, 입력 224×224, Batch 32, Seed 42, 증강 Train, 원본 Validation/Test
- 변경 조건: 2단계에서 Backbone 뒷부분을 학습 가능하게 변경
- 필수 동반 변경: 미세조정 학습률을 `1e-5`로 낮춤
- 선택 기준: 두 단계에서 생성된 모델 중 Validation Accuracy가 높은 모델
- Test: 최종 모델 선택이 끝난 뒤 한 번만 평가

> 이전 69.10% 결과는 데이터 분할이 달라 직접적인 성능 증감 비교에 사용하지 않습니다. 새 정리 데이터의 첫 결과가 새로운 기준선입니다.


## 1. Colab GPU와 Drive 준비

Colab 메뉴에서 GPU 런타임을 선택하고 `모두 실행`합니다. Drive 연결 창이 나타나면 승인합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install seaborn scikit-learn

import hashlib
import json
import platform
import random
import shutil
import zipfile
from datetime import datetime
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import EfficientNetB0

print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('GPU가 없습니다. Colab 런타임 유형을 GPU로 변경하세요.')


## 2. 실험 설정

이번에 생성된 ZIP 경로를 기본값으로 넣었습니다. 파일을 옮겼을 때만 `DATA_ZIP_PATH`를 수정합니다.


In [ ]:
MY_DRIVE = Path('/content/drive/MyDrive')
DATA_ZIP_PATH = MY_DRIVE / 'mediflow_datasets' / 'hair_clean_v1_20260907_053616.zip'
EXPECTED_DATA_SHA256 = '2ac7260663cf69835ba50edb6ae8c7e7ac13be9c73b9f7ea24f60e0342e7e156'
CLASS_NAMES = ['모낭사이홍반', '미세각질', '비듬', '탈모', '피지과다']
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
HEAD_EPOCHS = 15
HEAD_LEARNING_RATE = 1e-4
FINE_TUNE_EPOCHS = 10
FINE_TUNE_LEARNING_RATE = 1e-5
UNFREEZE_LAST_N = 30
SEED = 42
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
LOCAL_ROOT = Path('/content') / f'hair_clean_training_{RUN_ID}'
LOCAL_ZIP_PATH = LOCAL_ROOT / DATA_ZIP_PATH.name
EXTRACT_ROOT = LOCAL_ROOT / 'dataset'
RESULT_ROOT = LOCAL_ROOT / 'results'
DRIVE_RESULT_ROOT = MY_DRIVE / 'mediflow_experiments' / 'hair' / f'clean_two_stage_{RUN_ID}'
AUTOTUNE = tf.data.AUTOTUNE

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
LOCAL_ROOT.mkdir(parents=True, exist_ok=False)
EXTRACT_ROOT.mkdir()
RESULT_ROOT.mkdir()
print('실험 ID:', RUN_ID)
print('데이터:', DATA_ZIP_PATH)
print('결과:', DRIVE_RESULT_ROOT)


## 3. 데이터 ZIP 검증과 압축 해제

Drive에서 Colab으로 복사하면서 SHA-256을 계산해 방금 만든 데이터와 같은 파일인지 확인합니다.


In [ ]:
def copy_with_sha256(source, destination):
    digest = hashlib.sha256()
    with source.open('rb') as src, destination.open('wb') as dst:
        while True:
            chunk = src.read(8 * 1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
            dst.write(chunk)
    return digest.hexdigest()

def safe_extract(zip_path, destination):
    destination_resolved = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination_resolved / member.filename).resolve()
            if destination_resolved not in target.parents and target != destination_resolved:
                raise ValueError(f'안전하지 않은 ZIP 경로: {member.filename}')
        archive.extractall(destination_resolved)

if not DATA_ZIP_PATH.is_file():
    raise FileNotFoundError(f'정리 데이터 ZIP을 찾지 못했습니다: {DATA_ZIP_PATH}')
DATA_SHA256 = copy_with_sha256(DATA_ZIP_PATH, LOCAL_ZIP_PATH)
if DATA_SHA256 != EXPECTED_DATA_SHA256:
    raise ValueError(f'데이터 ZIP 식별값이 다릅니다. 예상={EXPECTED_DATA_SHA256}, 실제={DATA_SHA256}')
safe_extract(LOCAL_ZIP_PATH, EXTRACT_ROOT)
print('데이터 ZIP 검증 및 압축 해제 완료')


## 4. 데이터 구조와 수량 확인

클래스 순서는 기존 결과 계약과 동일하게 고정합니다. 모델 내부에 `Rescaling(1/255)`이 있으므로 외부 `/255.0` 정규화는 하지 않습니다.


In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def find_dataset_root():
    matches = [
        path for path in EXTRACT_ROOT.rglob('augmented')
        if path.is_dir() and all((path / split).is_dir() for split in ('train', 'val', 'test'))
    ]
    if len(matches) != 1:
        raise ValueError(f'augmented 데이터 폴더를 하나로 확정할 수 없습니다: {matches}')
    return matches[0]

def image_files(folder):
    return sorted(path for path in folder.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)

DATA_ROOT = find_dataset_root()
found_classes = sorted(path.name for path in (DATA_ROOT / 'train').iterdir() if path.is_dir())
if found_classes != sorted(CLASS_NAMES):
    raise ValueError(f'클래스 구성이 예상과 다릅니다: {found_classes}')
counts = {
    split: {class_name: len(image_files(DATA_ROOT / split / class_name)) for class_name in CLASS_NAMES}
    for split in ('train', 'val', 'test')
}
for split in counts:
    if any(value == 0 for value in counts[split].values()):
        raise ValueError(f'{split}에 이미지가 없는 클래스가 있습니다: {counts[split]}')
display(pd.DataFrame(counts).rename_axis('class'))
print('클래스 순서:', CLASS_NAMES)


## 5. TensorFlow 데이터셋과 클래스 가중치

충돌 사진 제거 후 클래스별 수량이 달라질 수 있어 Train 수량의 역비율로 클래스 가중치를 계산합니다. 두 학습 단계에 같은 가중치를 사용합니다.


In [ ]:
def make_dataset(split, shuffle):
    dataset = tf.keras.utils.image_dataset_from_directory(
        DATA_ROOT / split,
        labels='inferred',
        label_mode='int',
        class_names=CLASS_NAMES,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED if shuffle else None,
    )
    return dataset.prefetch(AUTOTUNE)

train_ds = make_dataset('train', True)
val_ds = make_dataset('val', False)
test_ds = make_dataset('test', False)
train_labels = np.concatenate([
    np.full(counts['train'][class_name], index, dtype=np.int64)
    for index, class_name in enumerate(CLASS_NAMES)
])
weights = compute_class_weight(class_weight='balanced', classes=np.arange(len(CLASS_NAMES)), y=train_labels)
CLASS_WEIGHT = {index: float(weight) for index, weight in enumerate(weights)}
print('클래스 가중치:', CLASS_WEIGHT)


## 6. 1단계 — 분류 부분 학습

EfficientNet-B0의 ImageNet 특징 추출 부분을 고정합니다. Validation Accuracy가 가장 높은 모델을 저장합니다.


In [ ]:
def build_head_model():
    backbone = EfficientNetB0(
        include_top=False, weights='imagenet',
        input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
    )
    backbone.trainable = False
    inputs = keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
    features = backbone(inputs, training=False)
    features = keras.layers.GlobalAveragePooling2D()(features)
    features = keras.layers.Dropout(0.3)(features)
    outputs = keras.layers.Dense(len(CLASS_NAMES), activation='softmax')(features)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(HEAD_LEARNING_RATE),
        loss='sparse_categorical_crossentropy', metrics=['accuracy'],
    )
    return model

HEAD_MODEL_PATH = RESULT_ROOT / 'stage1_head_best.keras'
head_model = build_head_model()
head_history = head_model.fit(
    train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, class_weight=CLASS_WEIGHT,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            str(HEAD_MODEL_PATH), monitor='val_accuracy', mode='max', save_best_only=True, verbose=1
        ),
        keras.callbacks.CSVLogger(str(RESULT_ROOT / 'stage1_training_log.csv')),
    ],
    verbose=1,
)
head_best_epoch = int(np.argmax(head_history.history['val_accuracy'])) + 1
head_best_val = float(max(head_history.history['val_accuracy']))
print('1단계 Best Epoch:', head_best_epoch)
print('1단계 Best Validation Accuracy:', head_best_val)


## 7. 2단계 — 부분 미세조정

1단계 최고 모델에서 이어서 EfficientNet 마지막 30개 계층을 엽니다. Batch Normalization은 계속 고정합니다.


In [ ]:
FINE_MODEL_PATH = RESULT_ROOT / 'stage2_finetune_best.keras'
fine_model = keras.models.load_model(HEAD_MODEL_PATH, compile=False)
backbone_candidates = [
    layer for layer in fine_model.layers
    if isinstance(layer, keras.Model) and 'efficientnet' in layer.name.lower()
]
if len(backbone_candidates) != 1:
    raise ValueError(f'EfficientNet Backbone을 하나로 찾지 못했습니다: {[x.name for x in backbone_candidates]}')
backbone = backbone_candidates[0]
backbone.trainable = True
for layer in backbone.layers[:-UNFREEZE_LAST_N]:
    layer.trainable = False
for layer in backbone.layers[-UNFREEZE_LAST_N:]:
    layer.trainable = not isinstance(layer, keras.layers.BatchNormalization)
fine_model.compile(
    optimizer=keras.optimizers.Adam(FINE_TUNE_LEARNING_RATE),
    loss='sparse_categorical_crossentropy', metrics=['accuracy'],
)
trainable_backbone_layers = [layer.name for layer in backbone.layers if layer.trainable]
print('학습 가능한 Backbone 계층:', trainable_backbone_layers)
fine_history = fine_model.fit(
    train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, class_weight=CLASS_WEIGHT,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            str(FINE_MODEL_PATH), monitor='val_accuracy', mode='max', save_best_only=True, verbose=1
        ),
        keras.callbacks.CSVLogger(str(RESULT_ROOT / 'stage2_training_log.csv')),
    ],
    verbose=1,
)
fine_best_epoch = int(np.argmax(fine_history.history['val_accuracy'])) + 1
fine_best_val = float(max(fine_history.history['val_accuracy']))
print('2단계 Best Epoch:', fine_best_epoch)
print('2단계 Best Validation Accuracy:', fine_best_val)


## 8. Validation으로 최종 모델 선택 후 Test 평가

1단계와 2단계의 Validation Accuracy만 비교해 최종 모델을 정합니다. 그 다음 Test를 한 번 평가합니다.


In [ ]:
if fine_best_val > head_best_val:
    selected_stage = 'stage2_partial_finetuning'
    selected_source_path = FINE_MODEL_PATH
    selected_val_accuracy = fine_best_val
else:
    selected_stage = 'stage1_head_only'
    selected_source_path = HEAD_MODEL_PATH
    selected_val_accuracy = head_best_val
FINAL_MODEL_PATH = RESULT_ROOT / 'best_model.keras'
shutil.copy2(selected_source_path, FINAL_MODEL_PATH)
selected_model = keras.models.load_model(FINAL_MODEL_PATH, compile=False)

y_true, y_pred = [], []
for images, labels in test_ds:
    probabilities = selected_model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(probabilities, axis=1).tolist())
y_true = np.asarray(y_true)
y_pred = np.asarray(y_pred)
test_accuracy = float(accuracy_score(y_true, y_pred))
macro_f1 = float(f1_score(y_true, y_pred, average='macro'))
report = classification_report(
    y_true, y_pred, labels=list(range(len(CLASS_NAMES))), target_names=CLASS_NAMES,
    output_dict=True, zero_division=0
)
print('선택 모델:', selected_stage)
print('선택 Validation Accuracy:', selected_val_accuracy)
print('Test Accuracy:', test_accuracy)
print('Macro F1:', macro_f1)
print(classification_report(
    y_true, y_pred, labels=list(range(len(CLASS_NAMES))), target_names=CLASS_NAMES,
    digits=6, zero_division=0
))


## 9. 학습 및 평가 그래프

Accuracy/Loss 변화, Confusion Matrix, 클래스별 F1, 두 단계의 최고 Validation Accuracy를 저장합니다.


In [ ]:
head_epoch_axis = np.arange(1, len(head_history.history['accuracy']) + 1)
fine_epoch_axis = np.arange(
    len(head_history.history['accuracy']) + 1,
    len(head_history.history['accuracy']) + len(fine_history.history['accuracy']) + 1,
)
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(head_epoch_axis, head_history.history['accuracy'], label='Stage 1 Train')
axes[0].plot(head_epoch_axis, head_history.history['val_accuracy'], label='Stage 1 Validation')
axes[0].plot(fine_epoch_axis, fine_history.history['accuracy'], label='Stage 2 Train')
axes[0].plot(fine_epoch_axis, fine_history.history['val_accuracy'], label='Stage 2 Validation')
axes[0].axvline(HEAD_EPOCHS + 0.5, color='gray', linestyle='--')
axes[0].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[0].legend()
axes[0].grid(True)
axes[1].plot(head_epoch_axis, head_history.history['loss'], label='Stage 1 Train')
axes[1].plot(head_epoch_axis, head_history.history['val_loss'], label='Stage 1 Validation')
axes[1].plot(fine_epoch_axis, fine_history.history['loss'], label='Stage 2 Train')
axes[1].plot(fine_epoch_axis, fine_history.history['val_loss'], label='Stage 2 Validation')
axes[1].axvline(HEAD_EPOCHS + 0.5, color='gray', linestyle='--')
axes[1].set(title='Loss', xlabel='Epoch', ylabel='Loss')
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.savefig(RESULT_ROOT / 'training_curves.png', dpi=200, bbox_inches='tight')
plt.show()

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
plt.figure(figsize=(9, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Hair Clean Dataset Confusion Matrix')
plt.tight_layout()
plt.savefig(RESULT_ROOT / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

class_f1 = [float(report[name]['f1-score']) for name in CLASS_NAMES]
plt.figure(figsize=(10, 5))
bars = plt.bar(CLASS_NAMES, class_f1, color='steelblue')
plt.ylim(0, 1)
plt.ylabel('F1')
plt.title('Class F1')
for bar, value in zip(bars, class_f1):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 0.015, f'{value:.3f}', ha='center')
plt.tight_layout()
plt.savefig(RESULT_ROOT / 'class_f1.png', dpi=200, bbox_inches='tight')
plt.show()

plt.figure(figsize=(7, 5))
stage_values = [head_best_val, fine_best_val]
stage_bars = plt.bar(['Stage 1 Head', 'Stage 2 Fine-tune'], stage_values, color=['gray', 'seagreen'])
plt.ylim(0, 1)
plt.ylabel('Best Validation Accuracy')
for bar, value in zip(stage_bars, stage_values):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 0.015, f'{value:.4f}', ha='center')
plt.tight_layout()
plt.savefig(RESULT_ROOT / 'stage_validation_comparison.png', dpi=200, bbox_inches='tight')
plt.show()


## 10. 결과 기록 및 Drive 저장

모델, 원시 수치, 환경, 클래스 순서, 데이터 식별값과 그래프를 실행별 새 Drive 폴더에 저장합니다. 기존 결과는 삭제하거나 덮어쓰지 않습니다.


In [ ]:
head_history_json = {key: [float(value) for value in values] for key, values in head_history.history.items()}
fine_history_json = {key: [float(value) for value in values] for key, values in fine_history.history.items()}
config = {
    'experiment': 'hair_clean_two_stage_training',
    'hypothesis': '부분 미세조정이 동일한 정리 데이터의 head-only 모델보다 Validation 성능을 개선한다.',
    'data_zip_path': str(DATA_ZIP_PATH),
    'data_zip_sha256': DATA_SHA256,
    'class_names': CLASS_NAMES,
    'normal_class_included': False,
    'out_of_scope_or_ambiguous_input_handling': '현재 모델에서 검증되지 않음',
    'image_size': list(IMAGE_SIZE),
    'batch_size': BATCH_SIZE,
    'seed': SEED,
    'class_counts': counts,
    'class_weight': CLASS_WEIGHT,
    'stage1': {
        'epochs': HEAD_EPOCHS, 'learning_rate': HEAD_LEARNING_RATE,
        'best_epoch': head_best_epoch, 'best_val_accuracy': head_best_val,
    },
    'stage2': {
        'epochs': FINE_TUNE_EPOCHS, 'learning_rate': FINE_TUNE_LEARNING_RATE,
        'unfreeze_last_n': UNFREEZE_LAST_N, 'batch_normalization_frozen': True,
        'best_epoch': fine_best_epoch, 'best_val_accuracy': fine_best_val,
    },
    'selection_metric': 'val_accuracy',
    'selected_stage': selected_stage,
    'selected_val_accuracy': selected_val_accuracy,
    'test_accuracy': test_accuracy,
    'macro_f1': macro_f1,
    'result_interpretation': '새 정리 데이터의 최초 기준 결과; 이전 분할 결과와 직접 비교하지 않음',
    'environment': {
        'python': platform.python_version(), 'tensorflow': tf.__version__,
        'keras': keras.__version__, 'numpy': np.__version__,
    },
    'code_reference': {
        'local_repository_commit_at_notebook_creation': 'b5faa6d937229a49e9d62541a30e39f3b75a3c77',
        'notebook_state_at_creation': 'uncommitted',
    },
}
(RESULT_ROOT / 'training_config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
(RESULT_ROOT / 'stage1_history.json').write_text(json.dumps(head_history_json, ensure_ascii=False, indent=2), encoding='utf-8')
(RESULT_ROOT / 'stage2_history.json').write_text(json.dumps(fine_history_json, ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame(report).transpose().to_csv(RESULT_ROOT / 'classification_report.csv', encoding='utf-8-sig')
pd.DataFrame([
    {'stage': 'stage1_head_only', 'best_val_accuracy': head_best_val},
    {'stage': 'stage2_partial_finetuning', 'best_val_accuracy': fine_best_val},
]).to_csv(RESULT_ROOT / 'stage_comparison.csv', index=False, encoding='utf-8-sig')

if DRIVE_RESULT_ROOT.exists():
    raise FileExistsError(f'결과 폴더가 이미 있습니다. 덮어쓰지 않습니다: {DRIVE_RESULT_ROOT}')
DRIVE_RESULT_ROOT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(RESULT_ROOT, DRIVE_RESULT_ROOT)
print('Drive 저장 완료:', DRIVE_RESULT_ROOT)
for path in sorted(DRIVE_RESULT_ROOT.iterdir()):
    print(' -', path.name)


## 완료 후 확인할 값

마지막 셀의 Drive 결과 경로와 `선택 모델`, `선택 Validation Accuracy`, `Test Accuracy`, `Macro F1`을 확인합니다. Test 결과를 보고 같은 데이터에서 설정을 반복 변경하지 않습니다. 새 가설은 별도 실험으로 진행합니다.
